# 11b — Dictionary growth metrics

Tracks DICTIONARY_ENTRY and NORM nodes over time and writes a per-day growth artifact for the eval dashboard.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'apps').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
from apps.backend.graph.neo4j_client import get_driver

driver = get_driver()
with driver.session() as s:
    counts = s.run("""
        MATCH (d:DICTIONARY_ENTRY) WITH count(d) AS dict_n
        MATCH (n:NORM) RETURN dict_n, count(n) AS norm_n
    """).single()
snapshot = {'ts': datetime.now(timezone.utc).isoformat(), 'dictionary_entries': counts['dict_n'], 'norms': counts['norm_n']}
print(json.dumps(snapshot, indent=2))

ART = REPO_ROOT / 'notebooks' / '_artifacts' / '11b_dict_growth'
ART.mkdir(parents=True, exist_ok=True)
(ART / 'growth.json').write_text(json.dumps(snapshot, ensure_ascii=False, indent=2))
print('artifact:', ART / 'growth.json')
